# 문서 분할(청킹) 담당 (C) — tuning

**담당:** 문서 분할 전략 검토 및 성능 향상

**실험 변수:** 분할 전략(splitter) (baseline: `recursive` → tuning: `semantic`). LLM/embedding_model/chunk_size/top_k는 baseline과 동일하게 유지하세요.

In [1]:
# 1. 환경 설정
import os
import time
import pandas as pd
from dotenv import load_dotenv

load_dotenv()  # .env 파일에서 API 키 로드 (팀원 각자 자신의 .env 사용)

# 필요한 키 예시 (.env 파일에 아래처럼 넣어두세요)
# HUGGINGFACEHUB_API_TOKEN=hf_...
# UPSTAGE_API_KEY=up_...        (필요시)
# OPENAI_API_KEY=sk-...         (필요시)

print("환경 변수 로드 완료")

환경 변수 로드 완료


## CONFIG
아래 설정값이 이 실행(run)의 조건입니다.

In [2]:
# CONFIG — 문서 분할(청킹) 담당 (C) — tuning
CONFIG = {
    "run_name": "C_chunking_tuning",
    "llm_model": "Qwen/Qwen2.5-7B-Instruct",
    "temperature": 0,
    "embedding_model": "jhgan/ko-sroberta-multitask",  # 팀 공통 baseline (B 담당 축 — 고정)
    "splitter": "semantic",           # recursive(baseline) → semantic(tuning) — C 담당 변수
    "chunk_size": 500,
    "chunk_overlap": 50,
    "semantic_breakpoint_type": "percentile",     # splitter=semantic일 때만 사용
    "semantic_breakpoint_amount": 95,             # splitter=semantic일 때만 사용
    "search_type": "similarity",
    "search_kwargs": {'k': 3},
    "prompt_template": "다음 문맥을 근거로 질문에 답하세요. 문맥에 없는 내용은 모른다고 답하세요.\n[문맥]\n{context}\n\n[질문]\n{question}",
}

for k, v in CONFIG.items():
    print(f"{k}: {v}")

run_name: C_chunking_tuning
llm_model: Qwen/Qwen2.5-7B-Instruct
temperature: 0
embedding_model: jhgan/ko-sroberta-multitask
splitter: semantic
chunk_size: 500
chunk_overlap: 50
semantic_breakpoint_type: percentile
semantic_breakpoint_amount: 95
search_type: similarity
search_kwargs: {'k': 3}
prompt_template: 다음 문맥을 근거로 질문에 답하세요. 문맥에 없는 내용은 모른다고 답하세요.
[문맥]
{context}

[질문]
{question}


## 1. PDF 경로

In [3]:
# 2. PDF 경로 설정
# ⚠️ TODO: 본인 PC에 있는 자동차관리법 조문 PDF 경로로 수정하세요.
PDF_PATH = "./data/자동차관리법.pdf"

assert os.path.exists(PDF_PATH), f"PDF 파일을 찾을 수 없습니다: {PDF_PATH} (경로를 확인하세요)"
print(f"PDF 경로 확인 완료: {PDF_PATH}")

PDF 경로 확인 완료: ./data/자동차관리법.pdf


## 2. 문서 로드

In [4]:
# 3. 문서 로드 (청킹은 다음 셀에서 splitter별로 분기)
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()
print(f"로드된 페이지 수: {len(docs)}")

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_30668\1310291269.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


로드된 페이지 수: 97


## 3. 임베딩 모델 로드
`splitter=semantic`은 임베딩 유사도로 문서를 자르기 때문에, 청킹보다 임베딩을 먼저 로드합니다 (벡터스토어 구축 때도 같은 임베딩 객체를 재사용합니다).

In [5]:
# 4. 임베딩 모델 로드
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name=CONFIG["embedding_model"],
    model_kwargs={"device": "cpu"},   # GPU는 LLM(4bit)이 통째로 쓰게 비워둠
)
print(f"임베딩 모델 로드 완료: {CONFIG['embedding_model']}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

임베딩 모델 로드 완료: jhgan/ko-sroberta-multitask


## 4. 문서 분할 (청킹) — C 담당 변수

In [6]:
# 5. 분할 전략별 TextSplitter 구성 후 청킹 실행
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter,
)

def build_splitter(cfg, embeddings=None):
    kind = cfg["splitter"]
    if kind == "recursive":
        return RecursiveCharacterTextSplitter(chunk_size=cfg["chunk_size"], chunk_overlap=cfg["chunk_overlap"])
    elif kind == "character":
        return CharacterTextSplitter(chunk_size=cfg["chunk_size"], chunk_overlap=cfg["chunk_overlap"], separator="\n")
    elif kind == "token":
        return TokenTextSplitter(chunk_size=cfg["chunk_size"], chunk_overlap=cfg["chunk_overlap"])
    elif kind == "semantic":
        # 임베딩 기반 의미 단위 분할 — langchain_experimental 필요
        # (유지보수 종료(sunset) 예정 패키지라 추후 대체 라이브러리 검토 필요)
        from langchain_experimental.text_splitter import SemanticChunker
        if embeddings is None:
            raise ValueError("semantic 분할은 embeddings 객체가 필요합니다.")
        return SemanticChunker(
            embeddings,
            breakpoint_threshold_type=cfg.get("semantic_breakpoint_type", "percentile"),
            breakpoint_threshold_amount=cfg.get("semantic_breakpoint_amount", 95),
        )
    else:
        raise ValueError(f"알 수 없는 splitter: {kind}")

splitter = build_splitter(CONFIG, embeddings=embeddings)
splits = splitter.split_documents(docs)
print(
    f"청킹 완료: {len(splits)}개 조각 "
    f"(splitter={CONFIG['splitter']}, chunk_size={CONFIG.get('chunk_size')}, overlap={CONFIG.get('chunk_overlap')})"
)

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_30668\496341097.py:19: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


청킹 완료: 384개 조각 (splitter=semantic, chunk_size=500, overlap=50)


## 5. 벡터스토어

In [7]:
# 6. 벡터스토어 구축 (임베딩은 위에서 만든 객체 재사용)
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name=CONFIG["run_name"],
)
print("벡터스토어 구축 완료 (Chroma)")

벡터스토어 구축 완료 (Chroma)


## 6. Retriever

In [8]:
# 7. Retriever 설정
retriever = vectorstore.as_retriever(
    search_type=CONFIG["search_type"],
    search_kwargs=CONFIG["search_kwargs"],
)
print(f"Retriever 설정 완료: search_type={CONFIG['search_type']}, kwargs={CONFIG['search_kwargs']}")

Retriever 설정 완료: search_type=similarity, kwargs={'k': 3}


## 7. LLM

In [9]:
# 8. LLM 로드 (로컬 GPU에 직접 다운로드해서 실행 — HF Inference API/OpenAI 크레딧 문제 회피)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(CONFIG["llm_model"])
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["llm_model"],
    quantization_config=quantization_config,
    device_map="auto",
)

text_gen_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=CONFIG["temperature"] > 0,
    **({"temperature": CONFIG["temperature"]} if CONFIG["temperature"] > 0 else {}),
    return_full_text=False,
)

llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=text_gen_pipeline))
print(f"LLM 로드 완료 (로컬 GPU, 4bit): {CONFIG['llm_model']}")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM 로드 완료 (로컬 GPU, 4bit): Qwen/Qwen2.5-7B-Instruct


## 8. RAG 체인

In [10]:
# 9. RAG 체인 구성
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

prompt = ChatPromptTemplate.from_template(CONFIG["prompt_template"])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print("RAG 체인 구성 완료")

RAG 체인 구성 완료


## 9. 평가 데이터셋 (팀 공통)

In [11]:
# 10. 평가 데이터셋 (팀 공통 baseline과 동일한 질문 10개)
eval_data = [
    {
        "category": "튜닝 관련",
        "question": "자동차 구조 및 장치 변경(튜닝)을 할 때 사전에 승인을 받아야 하는 항목과, 승인 없이 자유롭게 변경할 수 있는 경미한 사항에는 어떤 것들이 있나요?",
        "ground_truth": "자동차관리법 제34조에 따라 자동차의 구조 및 장치 중 대통령령으로 정하는 장치를 변경하려는 경우에는 시장·군수·구청장의 승인을 받아야 하며, 국토교통부령으로 정하는 경미한 사항의 변경은 승인 대상에서 제외됩니다."
    },
    {
        "category": "불법 튜닝 및 처벌",
        "question": "인증받지 않은 LED 전조등을 임의로 설치하거나 자동차 등화장치를 임의로 개조했을 때 적용되는 처벌 조항은 무엇인가요?",
        "ground_truth": "자동차관리법 제81조(벌칙)에 따라 시장·군수·구청장의 승인 없이 구조 및 장치를 변경한 자는 1년 이하의 징역 또는 1천만원 이하의 벌금에 처해질 수 있습니다."
    },
    {
        "category": "정기검사 주기 및 과태료",
        "question": "비사업용 승용자동차의 최초 등록 후 정기검사(종합검사) 유효기간은 어떻게 되며, 검사 기간을 경과했을 때 부과되는 과태료 기준은 무엇인가요?",
        "ground_truth": "자동차관리법 제43조 및 제84조에 따라 비사업용 승용자동차의 최초 정기검사 유효기간은 신차 등록 후 4년이며, 이후에는 2년마다 받아야 합니다. 검사 기간을 경과한 경우 경과 기간에 따라 최고 30만원 이하의 과태료가 부과됩니다."
    },
    {
        "category": "중고차 성능·상태 점검",
        "question": "중고차를 매매할 때 매매업자가 발급해 주는 '자동차 성능·상태 점검기록부'의 보증 기간과 보증 범위는 법적으로 어떻게 규정되어 있나요?",
        "ground_truth": "자동차관리법 제58조 및 관련 시행규칙에 따라 중고자동차 매매업자는 매수인에게 성능·상태 점검기록부를 발급해야 하며, 보증 기간은 30일 또는 주행거리 2,000km 이상으로 정하여 엔진·변속기 등 주요 부품의 결함을 보증해야 합니다."
    },
    {
        "category": "자동차 등록 번호판",
        "question": "자동차 등록번호판을 고의로 가리거나 알아보기 곤란하게 만든 경우, 자동차관리법에 따라 어떤 제재나 벌금을 받게 되나요?",
        "ground_truth": "자동차관리법 제81조에 따라 자동차 등록번호판을 가리거나 알아보기 곤란하게 고의로 운행한 자는 1년 이하의 징역 또는 1천만원 이하의 벌금에 처해집니다."
    },
    {
        "category": "말소 등록 (폐차 등)",
        "question": "자동차를 폐차하거나 말소 등록을 해야 할 때, 의무적으로 말소 신청을 해야 하는 법적 기한(사유 발생일로부터 며칠 이내)은 얼마인가요?",
        "ground_truth": "자동차관리법 제13조에 따라 자동차를 멸실하거나 해체(폐차)한 때에는 사유가 발생한 날부터 1개월 이내에 시·도지사에게 말소등록을 신청하여야 합니다."
    },
    {
        "category": "자율주행차 및 임시운행허가",
        "question": "자율주행자동차를 일반 도로에서 시험 운행하기 위해 임시운행허가를 받으려면 어떤 요건과 절차를 거쳐야 하거나 법적 근거는 무엇인가요?",
        "ground_truth": "자동차관리법 제27조 및 제27조의2에 따라 자율주행자동차를 시험·연구 목적으로 운행하려는 자는 국토교통부령으로 정하는 요건(안전운행요건 등)을 갖추어 국토교통부장관의 임시운행허가를 받아야 합니다."
    },
    {
        "category": "명의이전(이전등록)",
        "question": "중고차를 매매한 후 양수인(구매자)이 명의이전 등록을 하지 않을 경우, 법적인 이전 신청 기한과 이에 따른 과태료는 어떻게 되나요?",
        "ground_truth": "자동차관리법 제12조에 따라 이전등록 신청은 매매 등의 사유가 발생한 날부터 양수인이 일정 기한(이전등록 신청 기간) 내에 신청해야 하며, 이를 위반할 경우 지연 기간에 따라 최고 50만원 이하의 과태료가 부과됩니다."
    },
    {
        "category": "제작결함 시정(리콜)",
        "question": "자동차 제작자(제작사)가 차량의 제작결함(리콜 대상)을 발견했을 때, 소유자에게 시정방법 등을 통지하고 시정(리콜)을 이행해야 하는 법적 의무는 무엇인가요?",
        "ground_truth": "자동차관리법 제31조에 따라 제작자 등은 자동차에 결함이 있음을 발견한 경우 지체 없이 그 사실을 공개하고 시정계획서를 수립하여 국토교통부장관에게 보고한 후 소유자에게 통지하고 시정조치(리콜)를 해야 합니다."
    },
    {
        "category": "이륜자동차(오토바이) 관리",
        "question": "일정 기준 이상의 이륜자동차도 자동차관리법상 정기검사나 의무보험 가입 대상에 포함되나요? 관련 규정을 설명해 주세요.",
        "ground_truth": "자동차관리법 제43조 및 의무보험 관련 규정에 따라 일정 배기량 이상의 대형·형식 승인된 이륜자동차는 정기검사 대상 및 의무보험 가입 대상에 포함되어 법적 규제를 받습니다."
    },
]

## 10. 실행

In [12]:
# 11. 답변 생성 + 응답시간 측정
results = []
for item in eval_data:
    q = item["question"]
    start = time.time()
    answer = rag_chain.invoke(q)
    elapsed = time.time() - start
    results.append({
        "category": item["category"],
        "question": q,
        "ground_truth": item["ground_truth"],
        "answer": answer,
        "response_time_sec": round(elapsed, 3),
    })
    print(f"[{elapsed:.2f}s] Q: {q}\n -> A: {answer[:80]}...\n")

df_results = pd.DataFrame(results)
df_results.head()

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the docum

[593.21s] Q: 자동차 구조 및 장치 변경(튜닝)을 할 때 사전에 승인을 받아야 하는 항목과, 승인 없이 자유롭게 변경할 수 있는 경미한 사항에는 어떤 것들이 있나요?
 -> A: 자동차의 구조 및 장치를 변경(튜닝)할 때 사전에 승인을 받아야 하는 항목은 다음과 같습니다:

1. 제34조(자동차의 튜닝) ①항에 따라, 자...



[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[317.59s] Q: 인증받지 않은 LED 전조등을 임의로 설치하거나 자동차 등화장치를 임의로 개조했을 때 적용되는 처벌 조항은 무엇인가요?
 -> A: 제35조제2항을 위반한 경우에 해당합니다. 이 조항은 "자동차의 안전운행에 영향을 줄 수 있는 소프트웨어를 임의로 변경, 설치, 추가 또는 삭제...



[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[362.03s] Q: 중고차를 매매한 후 양수인(구매자)이 명의이전 등록을 하지 않을 경우, 법적인 이전 신청 기한과 이에 따른 과태료는 어떻게 되나요?
 -> A: 문맥에서 명시적으로 중고차 매매 후 양수인(구매자)이 명의 이전 등록을 하지 않을 경우의 법적인 이전 신청 기한과 과태료에 대한 정보는 없습니다...



[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[449.96s] Q: 자동차 제작자(제작사)가 차량의 제작결함(리콜 대상)을 발견했을 때, 소유자에게 시정방법 등을 통지하고 시정(리콜)을 이행해야 하는 법적 의무는 무엇인가요?
 -> A: 자동차 제작자가 차량의 제작결함(리콜 대상)을 발견했을 때, 소유자에게 시정방법 등을 통지하고 시정(리콜)을 이행해야 하는 법적 의무는 다음과 ...

[191.68s] Q: 일정 기준 이상의 이륜자동차도 자동차관리법상 정기검사나 의무보험 가입 대상에 포함되나요? 관련 규정을 설명해 주세요.
 -> A: 문맥에서 이와 관련된 명확한 정보가 없습니다. 따라서 이와 관련된 내용에 대해 알 수 없으므로, 이에 대한 답변을 드릴 수 없습니다....



,category,question,ground_truth,answer,response_time_sec
0,튜닝 관련,"자동차 구조 및 장치 변경(튜닝)을 할 때 사전에 승인을 받아야 하는 항목과, 승인...",자동차관리법 제34조에 따라 자동차의 구조 및 장치 중 대통령령으로 정하는 장치를 ...,자동차의 구조 및 장치를 변경(튜닝)할 때 사전에 승인을 받아야 하는 항목은 다음과...,593.207
1,불법 튜닝 및 처벌,인증받지 않은 LED 전조등을 임의로 설치하거나 자동차 등화장치를 임의로 개조했을 ...,자동차관리법 제81조(벌칙)에 따라 시장·군수·구청장의 승인 없이 구조 및 장치를 ...,"제35조제2항을 위반한 경우에 해당합니다. 이 조항은 ""자동차의 안전운행에 영향을 ...",317.588
2,정기검사 주기 및 과태료,"비사업용 승용자동차의 최초 등록 후 정기검사(종합검사) 유효기간은 어떻게 되며, 검...",자동차관리법 제43조 및 제84조에 따라 비사업용 승용자동차의 최초 정기검사 유효기...,문맥에서 제공된 정보로는 비사업용 승용자동차의 최초 등록 후 정기검사(종합검사)의 ...,343.381
3,중고차 성능·상태 점검,중고차를 매매할 때 매매업자가 발급해 주는 '자동차 성능·상태 점검기록부'의 보증 ...,자동차관리법 제58조 및 관련 시행규칙에 따라 중고자동차 매매업자는 매수인에게 성능...,법적으로 '자동차 성능·상태 점검기록부'에 대한 보증 기간과 보증 범위는 명확히 규...,519.046
4,자동차 등록 번호판,"자동차 등록번호판을 고의로 가리거나 알아보기 곤란하게 만든 경우, 자동차관리법에 따...",자동차관리법 제81조에 따라 자동차 등록번호판을 가리거나 알아보기 곤란하게 고의로 ...,자동차 등록번호판을 고의로 가리거나 알아보기 곤란하게 만드는 행위는 자동차관리법 제...,406.428


## 11. BERTScore 평가 (필수)

In [13]:
# 12. BERTScore 평가 (필수 지표)
from bert_score import score as bertscore

# 한국어는 klue/bert-base 사용 권장 (bert_score 라이브러리의 기본 레이어 매핑에는 없어 num_layers를 직접 지정)
P, R, F1 = bertscore(
    df_results["answer"].tolist(),
    df_results["ground_truth"].tolist(),
    model_type="klue/bert-base",
    num_layers=12,
    lang="ko",
    verbose=False,
)
df_results["bertscore_precision"] = P.tolist()
df_results["bertscore_recall"] = R.tolist()
df_results["bertscore_f1"] = F1.tolist()

print(f"평균 BERTScore F1: {df_results['bertscore_f1'].mean():.4f}")
print(f"평균 응답시간: {df_results['response_time_sec'].mean():.2f}초")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


평균 BERTScore F1: 0.5947
평균 응답시간: 438.06초


## 12. Hallucination 체크

In [14]:
# 13. Hallucination 체크 (팀원이 직접 답변을 읽고 표시)
# ⚠️ TODO: answer를 실제 조문 내용과 비교해서 사실과 다른 부분이 있으면 True로 표시하세요.
df_results["hallucination"] = False  # 기본값, 검토 후 True/False로 직접 수정

df_results

,category,question,ground_truth,answer,response_time_sec,bertscore_precision,bertscore_recall,bertscore_f1,hallucination
0,튜닝 관련,"자동차 구조 및 장치 변경(튜닝)을 할 때 사전에 승인을 받아야 하는 항목과, 승인...",자동차관리법 제34조에 따라 자동차의 구조 및 장치 중 대통령령으로 정하는 장치를 ...,자동차의 구조 및 장치를 변경(튜닝)할 때 사전에 승인을 받아야 하는 항목은 다음과...,593.207,0.662128,0.834931,0.738556,False
1,불법 튜닝 및 처벌,인증받지 않은 LED 전조등을 임의로 설치하거나 자동차 등화장치를 임의로 개조했을 ...,자동차관리법 제81조(벌칙)에 따라 시장·군수·구청장의 승인 없이 구조 및 장치를 ...,"제35조제2항을 위반한 경우에 해당합니다. 이 조항은 ""자동차의 안전운행에 영향을 ...",317.588,0.513465,0.535585,0.524292,False
2,정기검사 주기 및 과태료,"비사업용 승용자동차의 최초 등록 후 정기검사(종합검사) 유효기간은 어떻게 되며, 검...",자동차관리법 제43조 및 제84조에 따라 비사업용 승용자동차의 최초 정기검사 유효기...,문맥에서 제공된 정보로는 비사업용 승용자동차의 최초 등록 후 정기검사(종합검사)의 ...,343.381,0.586172,0.642944,0.613247,False
3,중고차 성능·상태 점검,중고차를 매매할 때 매매업자가 발급해 주는 '자동차 성능·상태 점검기록부'의 보증 ...,자동차관리법 제58조 및 관련 시행규칙에 따라 중고자동차 매매업자는 매수인에게 성능...,법적으로 '자동차 성능·상태 점검기록부'에 대한 보증 기간과 보증 범위는 명확히 규...,519.046,0.540359,0.601680,0.569373,False
4,자동차 등록 번호판,"자동차 등록번호판을 고의로 가리거나 알아보기 곤란하게 만든 경우, 자동차관리법에 따...",자동차관리법 제81조에 따라 자동차 등록번호판을 가리거나 알아보기 곤란하게 고의로 ...,자동차 등록번호판을 고의로 가리거나 알아보기 곤란하게 만드는 행위는 자동차관리법 제...,406.428,0.566449,0.797758,0.662494,False
5,말소 등록 (폐차 등),"자동차를 폐차하거나 말소 등록을 해야 할 때, 의무적으로 말소 신청을 해야 하는 법...",자동차관리법 제13조에 따라 자동차를 멸실하거나 해체(폐차)한 때에는 사유가 발생한...,"주어진 문맥에서 명확히 말소 신청의 법적 기한이 언급되어 있지 않습니다. 따라서, ...",279.215,0.476752,0.524987,0.499708,False
6,자율주행차 및 임시운행허가,자율주행자동차를 일반 도로에서 시험 운행하기 위해 임시운행허가를 받으려면 어떤 요건...,자동차관리법 제27조 및 제27조의2에 따라 자율주행자동차를 시험·연구 목적으로 운...,자율주행자동차를 일반 도로에서 시험 운행하기 위해 임시운행허가를 받으려면 다음과 같...,918.053,0.601150,0.760730,0.671590,False
7,명의이전(이전등록),"중고차를 매매한 후 양수인(구매자)이 명의이전 등록을 하지 않을 경우, 법적인 이전...",자동차관리법 제12조에 따라 이전등록 신청은 매매 등의 사유가 발생한 날부터 양수인...,문맥에서 명시적으로 중고차 매매 후 양수인(구매자)이 명의 이전 등록을 하지 않을 ...,362.029,0.526363,0.576987,0.550513,False
8,제작결함 시정(리콜),"자동차 제작자(제작사)가 차량의 제작결함(리콜 대상)을 발견했을 때, 소유자에게 시...",자동차관리법 제31조에 따라 제작자 등은 자동차에 결함이 있음을 발견한 경우 지체 ...,"자동차 제작자가 차량의 제작결함(리콜 대상)을 발견했을 때, 소유자에게 시정방법 등...",449.963,0.645951,0.751590,0.694778,False
9,이륜자동차(오토바이) 관리,일정 기준 이상의 이륜자동차도 자동차관리법상 정기검사나 의무보험 가입 대상에 포함되...,자동차관리법 제43조 및 의무보험 관련 규정에 따라 일정 배기량 이상의 대형·형식 ...,문맥에서 이와 관련된 명확한 정보가 없습니다. 따라서 이와 관련된 내용에 대해 알 ...,191.676,0.475275,0.380410,0.422584,False


## 13. 결과 저장

In [15]:
# 14. 결과 저장
os.makedirs("./eval", exist_ok=True)
save_path = f"./eval/results_{CONFIG['run_name']}.csv"
df_results.to_csv(save_path, index=False, encoding="utf-8-sig")
print(f"결과 저장 완료: {save_path}")

print("\n=== 요약 ===")
print(f"run_name        : {CONFIG['run_name']}")
print(f"LLM 모델        : {CONFIG['llm_model']}")
print(f"splitter        : {CONFIG['splitter']}")
print(f"chunk_size      : {CONFIG['chunk_size']} / overlap: {CONFIG['chunk_overlap']}")
print(f"search_type     : {CONFIG['search_type']} / kwargs: {CONFIG['search_kwargs']}")
print(f"평균 BERTScore F1 : {df_results['bertscore_f1'].mean():.4f}")
print(f"평균 응답시간     : {df_results['response_time_sec'].mean():.2f}초")

결과 저장 완료: ./eval/results_C_chunking_tuning.csv

=== 요약 ===
run_name        : C_chunking_tuning
LLM 모델        : Qwen/Qwen2.5-7B-Instruct
splitter        : semantic
chunk_size      : 500 / overlap: 50
search_type     : similarity / kwargs: {'k': 3}
평균 BERTScore F1 : 0.5947
평균 응답시간     : 438.06초
